# ✏️ Hands-On Paper & Pencil Lab: RMSNorm

Welcome! In this notebook, we calculate **RMSNorm (Root Mean Square Normalization)** by hand on paper using tiny, friendly numbers.

---

## 1. The Clean Formula

Given an input vector $x$ of dimension $d$:

$$\text{Step 1 (Root Mean Square):} \quad \text{RMS}(x) = \sqrt{ \frac{1}{d} \sum_{i=1}^d x_i^2 }$$

$$\text{Step 2 (Normalize & Scale):} \quad y_i = \frac{x_i}{\text{RMS}(x)} \times \gamma_i$$

*(Where $\gamma$ is the learnable scale parameter, initialized to $1.0$)*.

---
## 2. Your Pencil & Paper Exercise 📝

Suppose our token has a tiny vector of dimension $d = 2$:

$$x = [3.0, \; 4.0]$$

Let scale $\gamma = [1.0, \; 1.0]$.

### Follow these steps on paper:
1. **Square each number:**
   $$3.0^2 = \text{___}$$
   $$4.0^2 = \text{___}$$
2. **Find the average (mean) of the squares:**
   $$\text{Mean} = \frac{3.0^2 + 4.0^2}{2} = \text{___}$$
3. **Take the square root to get $\text{RMS}(x)$:**
   $$\text{RMS}(x) = \sqrt{\text{Mean}} = \text{___}$$
4. **Divide the original numbers by $\text{RMS}(x)$:**
   $$y_1 = \frac{3.0}{\text{RMS}(x)} = \text{___}$$
   $$y_2 = \frac{4.0}{\text{RMS}(x)} = \text{___}$$

---
## 3. Interactive Answer Checker 🎯

Type your paper-and-pencil answers into the boxes below and click **Check My Answers**!

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import ipywidgets as widgets
from IPython.display import display, Markdown
from src.model.norm import RMSNorm

# Input text widgets for student answers
rms_input = widgets.FloatText(value=0.0, description='RMS(x):', step=0.1)
y1_input = widgets.FloatText(value=0.0, description='Normed y1:', step=0.1)
y2_input = widgets.FloatText(value=0.0, description='Normed y2:', step=0.1)
check_btn = widgets.Button(description='Check My Answers', button_style='primary', icon='check')
output_box = widgets.Output()

def verify_answers(b):
    output_box.clear_output()
    with output_box:
        # True mathematical values
        # x = [3, 4] -> sum of sq = 9 + 16 = 25 -> mean = 12.5 -> sqrt(12.5) ~= 3.5355
        true_rms = ( (3.0**2 + 4.0**2) / 2.0 ) ** 0.5
        true_y1 = 3.0 / true_rms
        true_y2 = 4.0 / true_rms
        
        user_rms = rms_input.value
        user_y1 = y1_input.value
        user_y2 = y2_input.value
        
        rms_ok = abs(user_rms - true_rms) < 0.05
        y1_ok = abs(user_y1 - true_y1) < 0.05
        y2_ok = abs(user_y2 - true_y2) < 0.05
        
        if rms_ok and y1_ok and y2_ok:
            display(Markdown(f"""
### 🎉 EXCELLENT WORK! ALL ANSWERS ARE CORRECT!
- $\\text{{RMS}}(x) = \\sqrt{{12.5}} \\approx \\mathbf{{{true_rms:.4f}}}$
- $y_1 = 3.0 / {true_rms:.4f} \\approx \\mathbf{{{true_y1:.4f}}}$
- $y_2 = 4.0 / {true_rms:.4f} \\approx \\mathbf{{{true_y2:.4f}}}$
"""))
        else:
            display(Markdown(f"""
### ⚠️ Close, check the calculations:
- **RMS(x):** {'✅ Correct' if rms_ok else f'❌ You entered {user_rms}, expected around {true_rms:.4f}'}
- **Normed y1:** {'✅ Correct' if y1_ok else f'❌ You entered {user_y1}, expected around {true_y1:.4f}'}
- **Normed y2:** {'✅ Correct' if y2_ok else f'❌ You entered {user_y2}, expected around {true_y2:.4f}'}
"""))

check_btn.on_click(verify_answers)
display(widgets.VBox([rms_input, y1_input, y2_input, check_btn, output_box]))

---
## 4. Run Our Real PyTorch `RMSNorm` Layer

Now let's pass this exact tensor $[3.0, 4.0]$ into our `RMSNorm` class from `src/model/norm.py` to see it compute the exact same numbers!

In [ ]:
x_tensor = torch.tensor([[3.0, 4.0]], dtype=torch.float32)
norm_layer = RMSNorm(dim=2, eps=1e-8)

y_pytorch = norm_layer(x_tensor)

print(f"Input x:               {x_tensor.tolist()[0]}")
print(f"PyTorch RMSNorm y:     {[round(val, 4) for val in y_pytorch.tolist()[0]]}")

# Check the Root Mean Square of the output y:
rms_of_output = (y_pytorch.pow(2).mean().sqrt()).item()
print(f"RMS of normalized y:   {rms_of_output:.4f} (Notice it is exactly 1.0!)")